In [48]:
import sys
import os

# localizar caminho automaticamente
import pandas as pd

from rdkit import RDConfig
sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))

import sascorer
from rdkit import Chem

In [49]:
# ===== CONFIGURAÇÕES =====
# Defina aqui os caminhos e parâmetros

# Diretório base
base_dir = '/Users/francisco/Documents/Scripts/DeNovo_PfHDAC1/LigBuilder_results/Build_PfHDAC1_HIGHDIV_MAY/result/process_PfHDAC1/'

# Arquivo de entrada
input_file = base_dir + 'dados_sem_duplicatas.csv'

# Arquivos de saída
output_file_completo = base_dir + 'dados_com_sa_score.csv'
output_file_filtrado = base_dir + 'dados_sa_score_filtrado.csv'

# Nome da coluna de SMILES
coluna_smiles = 'SMILES'

# Threshold do SA Score (moléculas com SA Score < threshold receberão valor 1)
threshold = 6.0
# =========================

In [50]:
# Carregar o dataset
df = pd.read_csv(input_file)

print(f"Dataset carregado: {len(df)} linhas")
df.head()

Dataset carregado: 331 linhas


,SMILES,Name,InChI,Formula,MW,LogP,pKd,Structure,Synthesize,Chemical
0,Sc1nnc(o1)[C@@H](C(=O)OCc1noc(c1)CC(=O)C)N,30/000000101,InChI=1S/C11H12N4O5S/c1-5(16)2-7-3-6(15-20-7)4...,C11H12N4O5S,312,0.08,5.96,2.64,3.70,-170
1,O[C@H](CNCc1cc(Cl)cc(c1)Cl)c1nnc(o1)S,30/000000132,InChI=1S/C11H11Cl2N3O2S/c12-7-1-6(2-8(13)3-7)4...,C11H11N3O2SCl2,319,3.20,5.67,-1.70,2.22,-150
2,N(C[C@@H]([NH3+])F)NCCCOC(=O)c1ccc(cc1)Cl,28/000000002,InChI=1S/C12H17ClFN3O2/c13-10-4-2-9(3-5-10)12(...,C12H18N3O2FCl,290,2.84,5.57,-0.75,2.22,-170
3,Sc1nnc(o1)[C@@H](C(=O)OCc1nocc1C/C=C\C)N,30/000000195,InChI=1S/C12H14N4O4S/c1-2-3-4-7-5-19-16-8(7)6-...,C12H14N4O4S,310,1.27,5.55,2.03,3.70,-170
4,OC/C=C\c1onc(c1)COC(=O)[C@H](c1nnc(o1)S)N,30/000000165,InChI=1S/C11H12N4O5S/c12-8(9-13-14-11(21)19-9)...,C11H12N4O5S,312,0.18,5.49,1.43,3.70,-170


In [51]:
# Função para calcular o SA Score
def calculate_sa_score(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            return sascorer.calculateScore(mol)
        else:
            return None
    except:
        return None

# Aplicar a função na coluna de SMILES
df['SA_Score'] = df[coluna_smiles].apply(calculate_sa_score)

# Criar coluna binária baseada no threshold
df['SA_Score_Binary'] = (df['SA_Score'] < threshold).astype(int)

# Visualizar resultado
df.head()

[15:54:57] Explicit valence for atom # 20 N, 4, is greater than permitted
[15:54:57] Explicit valence for atom # 11 N, 4, is greater than permitted
[15:54:57] Explicit valence for atom # 18 N, 4, is greater than permitted
[15:54:57] Explicit valence for atom # 8 N, 4, is greater than permitted
[15:54:57] Explicit valence for atom # 4 N, 4, is greater than permitted
[15:54:57] Explicit valence for atom # 19 N, 4, is greater than permitted
[15:54:57] Explicit valence for atom # 14 N, 4, is greater than permitted
[15:54:57] Explicit valence for atom # 4 N, 4, is greater than permitted


,SMILES,Name,InChI,Formula,MW,LogP,pKd,Structure,Synthesize,Chemical,SA_Score,SA_Score_Binary
0,Sc1nnc(o1)[C@@H](C(=O)OCc1noc(c1)CC(=O)C)N,30/000000101,InChI=1S/C11H12N4O5S/c1-5(16)2-7-3-6(15-20-7)4...,C11H12N4O5S,312,0.08,5.96,2.64,3.70,-170,3.772337,1
1,O[C@H](CNCc1cc(Cl)cc(c1)Cl)c1nnc(o1)S,30/000000132,InChI=1S/C11H11Cl2N3O2S/c12-7-1-6(2-8(13)3-7)4...,C11H11N3O2SCl2,319,3.20,5.67,-1.70,2.22,-150,3.347764,1
2,N(C[C@@H]([NH3+])F)NCCCOC(=O)c1ccc(cc1)Cl,28/000000002,InChI=1S/C12H17ClFN3O2/c13-10-4-2-9(3-5-10)12(...,C12H18N3O2FCl,290,2.84,5.57,-0.75,2.22,-170,3.273269,1
3,Sc1nnc(o1)[C@@H](C(=O)OCc1nocc1C/C=C\C)N,30/000000195,InChI=1S/C12H14N4O4S/c1-2-3-4-7-5-19-16-8(7)6-...,C12H14N4O4S,310,1.27,5.55,2.03,3.70,-170,4.252351,1
4,OC/C=C\c1onc(c1)COC(=O)[C@H](c1nnc(o1)S)N,30/000000165,InChI=1S/C11H12N4O5S/c12-8(9-13-14-11(21)19-9)...,C11H12N4O5S,312,0.18,5.49,1.43,3.70,-170,4.009701,1


In [52]:
# Contar quantos 1s e 0s na coluna binária
contagem = df['SA_Score_Binary'].value_counts().sort_index()
print(contagem)

SA_Score_Binary
0      8
1    323
Name: count, dtype: int64


In [53]:
# Salvar dataset completo com SA Score
df.to_csv(output_file_completo, index=False)
print(f"Dataset completo salvo em: {output_file_completo}")

Dataset completo salvo em: /Users/francisco/Documents/Scripts/DeNovo_PfHDAC1/LigBuilder_results/Build_PfHDAC1_HIGHDIV_MAY/result/process_PfHDAC1/dados_com_sa_score.csv


In [54]:
# Filtrar apenas moléculas com SA_Score_Binary = 1 (SA Score < threshold)
df_filtered = df[df['SA_Score_Binary'] == 1].copy()

print(f"Total de moléculas filtradas: {len(df_filtered)} (SA Score < {threshold})")

# Salvar o dataframe filtrado em CSV
df_filtered.to_csv(output_file_filtrado, index=False)

print(f"Arquivo filtrado salvo em: {output_file_filtrado}")

Total de moléculas filtradas: 323 (SA Score < 6.0)
Arquivo filtrado salvo em: /Users/francisco/Documents/Scripts/DeNovo_PfHDAC1/LigBuilder_results/Build_PfHDAC1_HIGHDIV_MAY/result/process_PfHDAC1/dados_sa_score_filtrado.csv
